# Ipotesi 1 — 03. Analisi: il modello usa davvero il video?

Gira in locale o su Colab senza GPU. Legge i `.jsonl` in `results/ipotesi_video/` e produce:
1. la **tabella principale** (accuratezza, accordo con C0, kappa, McNemar);
2. il **verdetto** secondo il criterio fissato prima dei run;
3. il **grafico** per la tesi (`results/ipotesi_video/figure/ipotesi1.png` e `.pdf`);
4. i dettagli: per nutriente e bias posizionale;
5. gli **esempi qualitativi**.

**Come leggere le metriche**
- *Accordo con C0*: % di domande con la **stessa lettera** del video vero. E' la metrica
  principale: se il modello risponde dal testo, togliere o scambiare il video non cambia le risposte.
- *Kappa di Cohen*: lo stesso accordo, corretto per il caso. Serve perche' un modello che
  risponde quasi sempre la stessa lettera avrebbe un accordo alto senza guardare nulla.
- *McNemar*: confronto appaiato delle accuratezze sulle stesse domande.
- *IC 95%*: intervallo di Wilson. Con 50 domande e' largo circa ±13 punti: differenze piccole
  non sono distinguibili dal rumore.

In [ ]:
import json, sys
from pathlib import Path

try:                                    # su Colab: cartella su Drive
    from google.colab import drive
    drive.mount("/content/drive")
    RADICE = Path("/content/drive/MyDrive/Progetto Deep Learning 1")
except ImportError:                     # in locale: la radice del progetto
    RADICE = Path.cwd().resolve()
    while not (RADICE / "src" / "hdepic").exists() and RADICE != RADICE.parent:
        RADICE = RADICE.parent
sys.path.insert(0, str(RADICE / "src"))

import pandas as pd
from hdepic import eval as H
from hdepic import ablazione as A
pd.set_option("display.width", 200)

RISULTATI = RADICE / "results" / "ipotesi_video"
FIGURE = RADICE / "results" / "ipotesi_video" / "figure"
DERIVATI = RADICE / "data" / "derived" / "ipotesi_video"
MODELLI = ["videollama2", "llavavideo"]

ann = H.load_annotations(RADICE / "data" / "hd-epic", A.TASK)
swap_map = json.loads((DERIVATI / "swap_map.json").read_text())
print(f"criterio: accordo >= {A.SOGLIA_ACCORDO:.0%}, alpha = {A.ALPHA}")

## 1. Caricamento e allineamento delle domande

In [ ]:
df = A.carica_risultati(RISULTATI, MODELLI)
df = A.controlla_allineamento(df)

## 2. Tabella principale

In [ ]:
tab = A.tabella_principale(df)
tab.to_csv(RISULTATI / "ipotesi1_tabella_principale.csv", index=False)
tab

## 3. Verdetto

In [ ]:
esiti = A.verdetto(tab)

## 4. Grafico per la tesi

In [ ]:
FIGURE.mkdir(parents=True, exist_ok=True)
fig = A.grafico(tab, path=FIGURE / "ipotesi1")

## 5. Dettagli

**Per nutriente.** Se l'accuratezza senza video resta alta solo su alcuni nutrienti (per es. le
calorie), e' li' che la conoscenza del mondo basta a rispondere.

In [ ]:
A.per_nutriente(df)

**Bias posizionale.** Se senza video le risposte si concentrano su una lettera, il modello
ripiega su una preferenza di posizione invece che sul contenuto delle opzioni.

In [ ]:
A.bias_posizionale(df)

## 6. Esempi qualitativi

Domande in cui il modello da' **la stessa risposta** col video vero, senza video e col video di
un'altra ricetta. Sono gli esempi che rendono concreto il risultato nella tesi.

In [ ]:
for m in MODELLI:
    if m in set(df["modello"]):
        display(A.esempi_invariati(df, ann, swap_map, m, n=3))